In [48]:
from util_process import *

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
import os
import re
import numpy as np

In [50]:
# Group motion and range part files by person_day, sort by part number, then combine them into continuous files.
def combine_parts(motion_dir, range_dir, data_dir):
    
    motion_pattern = r"motion_part_\d+_label_regenerated_(.*?_routine_env_\d+_\w+)_parsed_grounded_.*?\.txt\.txt"
    range_pattern  = r"range_part_\d+_label_regenerated_(.*?_routine_env_\d+_\w+)_parsed_grounded_.*?\.txt"

    # =========================
    # Motion grouping
    # =========================
    motion_person_day_files = {}

    for filename in os.listdir(motion_dir):
        if filename.startswith("motion_part") and filename.endswith(".txt.txt"):
            match = re.search(motion_pattern, filename)
            if match:
                person_day = match.group(1)
                file_path = os.path.join(motion_dir, filename)

                if person_day not in motion_person_day_files:
                    motion_person_day_files[person_day] = []
                motion_person_day_files[person_day].append(file_path)

    for person_day in motion_person_day_files:
        motion_person_day_files[person_day].sort(
            key=lambda x: int(re.search(r"motion_part_(\d+)", os.path.basename(x)).group(1))
        )

    motion_file_groups = list(motion_person_day_files.values())

    for i, group in enumerate(motion_file_groups):
        print(f"\nMotion Group:")
        for file_path in group:
            print(f"  {os.path.basename(file_path)}")


    # =========================
    # Range grouping
    # =========================
    range_person_day_files = {}

    for filename in os.listdir(range_dir):
        if filename.startswith("range_part") and filename.endswith(".txt"):
            match = re.search(range_pattern, filename)
            if match:
                person_day = match.group(1)
                file_path = os.path.join(range_dir, filename)

                if person_day not in range_person_day_files:
                    range_person_day_files[person_day] = []
                range_person_day_files[person_day].append(file_path)

    for person_day in range_person_day_files:
        range_person_day_files[person_day].sort(
            key=lambda x: int(re.search(r"range_part_(\d+)", os.path.basename(x)).group(1))
        )

    range_file_groups = list(range_person_day_files.values())

    for i, group in enumerate(range_file_groups):
        print(f"\nRange Group:")
        for file_path in group:
            print(f"  {os.path.basename(file_path)}")


    # =========================
    # Combine range files
    # =========================
    for group in range_file_groups:
        first_file = os.path.basename(group[0])
        match = re.search(range_pattern, first_file)
        if match:
            person_day = match.group(1)

        combine_txt_files_with_frame_adjustment(group, f"{data_dir}/combined_range_sensors_{person_day}.txt")

    # =========================
    # Combine motion files
    # =========================
    for group in motion_file_groups:
        first_file = os.path.basename(group[0])
        match = re.search(motion_pattern, first_file)
        if match:
            person_day = match.group(1)

        combine_sensor_txt_files(group, f"{data_dir}/combined_motion_sensors_{person_day}.txt")
    
    print("[AgentSense] Finished combining motion/range parts into continuous files")


In [51]:
def process_to_npy():

    data_dir = DATA_DIR
    dest_dir = DEST_DIR

    os.makedirs(dest_dir, exist_ok=True)

    # Get motion files and create corresponding range files list
    motion_files = []
    range_files = []
    print("\n[AgentSense] Starting transfer to .npy files... ")
    for file in os.listdir(data_dir):
        if "motion" in file.lower() and file.endswith('.txt'):
            motion_files.append(f"{data_dir}/{file}")
            # Create corresponding range file name by replacing 'motion' with 'range'
            range_file = file.replace('motion', 'range')
            range_file = range_file.replace('labeled.txt', 'new_labeled.txt')
            range_files.append(f"{data_dir}/{range_file}")

    seg_states_all = []
    seg_sensors_all = []
    seg_timestamps_all = []
    aruba_seq_all = []
    milan_seq_all = []
    cairo_seq_all = []
    kyoto7_seq_all = []
    orange_seq_all = []
    # Dictionary to store timestamps for each motion file
    motion_file_timestamps = {}
    # Create dictionaries for each dataset's sequences
    aruba_file_sequences = {}
    milan_file_sequences = {}
    cairo_file_sequences = {}
    kyoto7_file_sequences = {}
    orange_file_sequences = {}

    for i in range(len(motion_files)):
        motion_file = motion_files[i]
        range_file = range_files[i]
        seg_states, seg_sensors, seg_timestamps, aruba_seq, milan_seq, cairo_seq, kyoto7_seq, orange_seq = process_virtual_data(motion_file, range_file)

        new_seg_states = []
        new_seg_sensors = []
        new_seg_timestamps = []
        new_aruba_seq = []
        new_milan_seq = []
        new_cairo_seq = []
        new_kyoto7_seq = []
        new_orange_seq = []
        # Store timestamps for this motion file
        motion_file_timestamps[motion_file] = []

        # Initialize sequence lists for this motion file
        aruba_file_sequences[motion_file] = []
        milan_file_sequences[motion_file] = []
        cairo_file_sequences[motion_file] = []
        kyoto7_file_sequences[motion_file] = []
        orange_file_sequences[motion_file] = []

        for j in range(len(seg_states)):
            if len(seg_states[j]) > 2 and None not in seg_timestamps[j]:
                new_seg_states.append(seg_states[j])
                new_seg_sensors.append(seg_sensors[j])
                new_seg_timestamps.append(seg_timestamps[j])
                new_aruba_seq.append(aruba_seq[j])
                new_milan_seq.append(milan_seq[j])
                new_cairo_seq.append(cairo_seq[j])
                new_kyoto7_seq.append(kyoto7_seq[j])
                new_orange_seq.append(orange_seq[j])

                # Store sequences corresponding to this motion file
                aruba_file_sequences[motion_file].append(aruba_seq[j])
                milan_file_sequences[motion_file].append(milan_seq[j])
                cairo_file_sequences[motion_file].append(cairo_seq[j])
                kyoto7_file_sequences[motion_file].append(kyoto7_seq[j])
                orange_file_sequences[motion_file].append(orange_seq[j])

                # Store timestamps corresponding to this motion file
                motion_file_timestamps[motion_file].append(seg_timestamps[j])

#                 if len(seg_states[j]) != len(seg_sensors[j]) or len(seg_states[j]) != len(seg_timestamps[j]):
#                     print(len(seg_states[j]))
#                     print(len(seg_sensors[j]))
#                     print(len(seg_timestamps[j]))

        seg_states_all.extend(new_seg_states)
        seg_sensors_all.extend(new_seg_sensors)
        seg_timestamps_all.extend(new_seg_timestamps)
        aruba_seq_all.extend(new_aruba_seq)
        milan_seq_all.extend(new_milan_seq)
        cairo_seq_all.extend(new_cairo_seq)
        kyoto7_seq_all.extend(new_kyoto7_seq)
        orange_seq_all.extend(new_orange_seq)


    # Convert to arrays of lists before saving
    seg_timestamps_arr = np.array([list(x) for x in seg_timestamps_all], dtype=object)
    seg_sensors_arr = np.array([list(x) for x in seg_sensors_all], dtype=object)
    seg_states_arr = np.array([list(x) for x in seg_states_all], dtype=object)
    orange_seq_arr = np.array(orange_seq_all, dtype=object)

    # Save segmented data
    np.save(f"{dest_dir}/virtual-aruba-x_time.npy", seg_timestamps_arr)
    np.save(f"{dest_dir}/virtual-aruba-x_sensor.npy", seg_sensors_arr)
    np.save(f"{dest_dir}/virtual-aruba-x_value.npy", seg_states_arr)

    np.save(f"{dest_dir}/virtual-milan-x_time.npy", seg_timestamps_arr)
    np.save(f"{dest_dir}/virtual-milan-x_sensor.npy", seg_sensors_arr)
    np.save(f"{dest_dir}/virtual-milan-x_value.npy", seg_states_arr)

    np.save(f"{dest_dir}/virtual-cairo-x_time.npy", seg_timestamps_arr)
    np.save(f"{dest_dir}/virtual-cairo-x_sensor.npy", seg_sensors_arr)
    np.save(f"{dest_dir}/virtual-cairo-x_value.npy", seg_states_arr)

    np.save(f"{dest_dir}/virtual-kyoto7-x_time.npy", seg_timestamps_arr)
    np.save(f"{dest_dir}/virtual-kyoto7-x_sensor.npy", seg_sensors_arr)
    np.save(f"{dest_dir}/virtual-kyoto7-x_value.npy", seg_states_arr)

    np.save(f"{dest_dir}/virtual-orange-x_time.npy", seg_timestamps_arr)
    np.save(f"{dest_dir}/virtual-orange-x_sensor.npy", seg_sensors_arr)
    np.save(f"{dest_dir}/virtual-orange-x_value.npy", seg_states_arr)

    # Convert sequences to arrays and save
    aruba_seq_arr = np.array(aruba_seq_all)
    milan_seq_arr = np.array(milan_seq_all)
    cairo_seq_arr = np.array(cairo_seq_all)
    kyoto7_seq_arr = np.array(kyoto7_seq_all)
    orange_seq_arr = np.array(orange_seq_all)

    np.save(f"{dest_dir}/virtual-aruba-y.npy", aruba_seq_arr)
    np.save(f"{dest_dir}/virtual-milan-y.npy", milan_seq_arr)
    np.save(f"{dest_dir}/virtual-cairo-y.npy", cairo_seq_arr)
    np.save(f"{dest_dir}/virtual-kyoto7-y.npy", kyoto7_seq_arr)
    np.save(f"{dest_dir}/virtual-orange-y.npy", orange_seq_arr)

    
    print(f"[AgentSense] All .npy files have been saved to {DEST_DIR}")


In [52]:
# =========================
# Configuration (User-editable)
# =========================

# Directory containing all motion location data files for a single character on a single day under 
# a specific home environment.
motion_dir = "data/step_10_motion_data"

# Directory containing all activity-to-frame-range data files for a single character on a single 
# day under a specific home environment.
range_dir = "data/step_10_range_data"

# Directory used to store combined motion and range files
data_dir = "data/step_11_combined_data"

# Directory to save the final segmented sensor datasets in .npy format
DEST_DIR = "data/final_npy_data"

# ================================================================================

# =========================
# Combine motion and range parts into full-day files
# =========================

# Generates combined motion and range sensor files in `data_dir`
combine_parts(motion_dir, range_dir, data_dir)

# =========================
# Convert combined sensor files into segmented datasets
# =========================

# The processing function reads combined files from DATA_DIR
DATA_DIR = data_dir

# Generates segmented sensor arrays and saves them to DEST_DIR
process_to_npy()



Motion Group:
  motion_part_1_label_regenerated_Sarah_routine_env_0_Monday_parsed_grounded_bedroom.txt.txt
  motion_part_2_label_regenerated_Sarah_routine_env_0_Monday_parsed_grounded_bathroom.txt.txt
  motion_part_3_label_regenerated_Sarah_routine_env_0_Monday_parsed_grounded_kitchen.txt.txt

Range Group:
  range_part_1_label_regenerated_Sarah_routine_env_0_Monday_parsed_grounded_bedroom.txt
  range_part_2_label_regenerated_Sarah_routine_env_0_Monday_parsed_grounded_bathroom.txt
  range_part_3_label_regenerated_Sarah_routine_env_0_Monday_parsed_grounded_kitchen.txt

[AgentSense] Combined activity-to-frame-range mapping files saved to: data/step_11_combined_data/combined_range_sensors_Sarah_routine_env_0_Monday.txt
[AgentSense] Combined motion location data written to: data/step_11_combined_data/combined_motion_sensors_Sarah_routine_env_0_Monday.txt
[AgentSense] Finished combining motion/range parts into continuous files

[AgentSense] Starting transfer to .npy files... 
[AgentSense] A